# BCI inefficiency

Hypothesis:
> Resting or pre-trial `alpha`/`mu` power predicts which subjects decode well.

## Datasets

- PhysioNet EEG Motor Movement/Imagery (EEGBCI):
Motor execution and imagery, fists/feet.
Used in notebook 06; large subject pool; both executed and imagined runs

- Cho2017:
Left/right hand imagery.
Many subjects for studying who can and cannot use MI

- Lee2019_MI (OpenBMI):
Left/right hand imagery.
Two sessions; large and recent; big download

Loader MOABB:
https://moabb.neurotechx.com/docs/dataset_summary.html

## Jupyter Server

```sh
apt install python3-full tmux
python3 -m venv .venv 
```

Jupyter daemon:
```sh
tmux new -s bci
# detach: ctrl-b + d
# attach: tmux attach-session -t bci
# sessions: tmux ls

. .venv/bin/activate

pip install notebook
jupyter notebook
# detach: ctrl-b + d
```

Get link with token:
```sh
. .venv/bin/activate
jupyter server list
```


## Project requirements

Every project must contain the following seven stages, each with at least one figure or table as evidence.

1. **Data audit.** Report subjects, sessions, runs, channels, sampling rate, units, reference, event codes and trial counts per class. Plot raw data and the sensor layout.
2. **Signal inspection.** Show the PSD before and after filtering, and justify filter type, cut-offs and any resampling (notebook 03).
3. **Artifact handling.** Apply at least one method from notebook 04 and show what it removed. Report how many channels, components or epochs were rejected, and why (notebook 05).
4. **Neurophysiological check.** Before decoding, show that the expected signal exists: the P300 in target-minus-non-target ERPs, or mu/beta desynchronization in band power over C3/C4.
5. **Features and classifier.** Build features and a classifier from the scope table inside one scikit-learn `Pipeline`, so that scaling and selection are fitted on training folds only.
6. **Honest evaluation.**
    - Split by run, session or subject with `GroupKFold` or leave-one-group-out, never by random epochs.
    - Report ROC AUC or balanced accuracy as mean ± SD across folds and per subject.
    - Estimate chance level with a permutation test or the binomial limit for your trial count.
    - Tune hyperparameters with nested CV or a held-out validation group.
7. **Hypothesis test.** Run your controlled comparison, changing one factor at a time, and test whether the difference exceeds fold-to-fold variability (for example, a paired Wilcoxon test across subjects).

**Reproducibility.** One notebook runs top to bottom on a fresh Colab CPU runtime. Pin package versions, set random seeds, and list the subject IDs used. Put shared code in functions rather than copied cells.


## MNE-Python

MNE-Python is an open-source Python package for data input/output, preprocessing, exploring, visualizing, and analyzing human neurophysiological data such as EEG.

In [ ]:
%pip install mne

  Using cached mne-1.13.2-py3-none-any.whl (7.7 MB)
  Using cached pooch-1.9.0-py3-none-any.whl (67 kB)
  Using cached decorator-5.3.1-py3-none-any.whl (10 kB)
  Using cached lazy_loader-0.6-py3-none-any.whl (8.8 kB)
  Using cached matplotlib-3.11.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.9 MB)
  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.9 MB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
  Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
  Using cached contourpy-1.3.3-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (355 kB)
  Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached fonttools-4.68.0-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (5.5 MB)
  Using cached kiwisolver-1.5.1-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (1.4 MB)
  Using cached pillow-13.0.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x8

In [2]:
import mne
print("mne:", mne.__version__) # 1.13.2

mne: 1.13.2


## EEGBCI

[/physionet-docs](https://physionet.org/content/eegmmidb/1.0.0/)

The EEG Motor Movement/Imagery Dataset provides 64-channel electroencephalography recordings collected with the BCI2000 system during real and imagined motor tasks. Participants completed 14 experimental runs, including two one-minute baseline recordings with eyes open and eyes closed, followed by three repetitions of four two-minute task conditions. ... EEG signals were recorded according to the international 10-10 electrode placement system at a sampling rate of 160 Hz and are provided in EDF+ format with accompanying annotation channels. Event labels identify rest periods and task onsets using three codes: T0 for rest, T1 for left-fist or both-fists movement/imagery depending on the run type, and T2 for right-fist or both-feet movement/imagery.

[/mne-docs](https://mne.tools/stable/generated/mne.datasets.eegbci.load_data.html)

| run | task |
| --- | --- |
| 1 | Baseline, eyes open |
| 2 | Baseline, eyes closed |
| 3, 7, 11 | Motor execution: left vs right hand |
| 4, 8, 12 | Motor imagery: left vs right hand |
| 5, 9, 13 | Motor execution: hands vs feet |
| 6, 10, 14 | Motor imagery: hands vs feet |



In [10]:
%pip install pooch

Note: you may need to restart the kernel to use updated packages.


In [12]:
import mne
from mne.datasets import eegbci

files = eegbci.load_data(
    subjects=[1, 2], # list(range(1, 110))
    runs=[6, 10, 14], # list(range(1, 15))
    path="./datasets",
    update_path=False # don't write path to MNE config
)

print(files)
# [Path('datasets/MNE-eegbci-data/files/eegmmidb/1.0.0/...edf'),]

Download complete in 08m32s (14.7 MB)
[PosixPath('datasets/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R06.edf'), PosixPath('datasets/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R10.edf'), PosixPath('datasets/MNE-eegbci-data/files/eegmmidb/1.0.0/S001/S001R14.edf'), PosixPath('datasets/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R06.edf'), PosixPath('datasets/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R10.edf'), PosixPath('datasets/MNE-eegbci-data/files/eegmmidb/1.0.0/S002/S002R14.edf')]


## Cho2017

BCI experiment for motor imagery movement (MI movement) of the left and right hands with 52 subjects (19 females, mean age ± SD age = 24.8 ± 3.86 years); Each subject took part in the same experiment, and subject ID was denoted and indexed as s1, s2, …, s52. Subjects s20 and s33 were both-handed, and the other 50 subjects were right-handed.

EEG data were collected using 64 Ag/AgCl active electrodes. A 64-channel montage based on the international 10-10 system was used to record the EEG signals with 512 Hz sampling rates.

In [4]:
%pip install moabb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 980.1/980.1 kB 2.7 MB/s eta 0:00:002.2 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 32.7 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 18.2 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 38.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.6/153.6 kB 27.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.5/260.5 kB 24.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.5/162.5 kB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 37.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 41.0 MB/s eta 0:00:00 MB/s eta 0:00:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 39.0 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━

In [18]:
from moabb.datasets import Cho2017

dataset = Cho2017()

files = dataset.data_path(
    subject=1,
    path="./datasets",
    update_path=False # don't write path to MNE config
)

print(files)

/home/john/Git/BCI_Midterm_Project/datasets/MNE-gigadb-data/gigadb-datasets/live/pub/10.5524/100001_101000/100295/mat_data/s01.mat
